# FIFA Men's World Cup 2026 Predictor
## Stage 3: Local Validation Loop & Baseline Model

**Author:** Kamweti Ryan Murunga  
**Objective:** Combine our Stage 2 features with our core training framework to establish a bulletproof cross-validation strategy.

### Methodology:
1. Merge the engineered historical feature store onto `Train.csv` and `Test.csv`.
2. Convert the categorical stages into numeric labels for training.
3. Build a 5-Fold Stratified Cross-Validation loop using LightGBM.
4. Export the baseline predictions formatted exactly to match `SampleSubmission.csv`.

In [1]:
!pip install lightgbm

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error, f1_score
from lightgbm import LGBMRegressor, LGBMClassifier
import warnings
warnings.filterwarnings('ignore')

# 1. ENFORCE PIPELINE REPRODUCIBILITY
SEED = 42
np.random.seed(SEED)

# 2. LOAD DATASETS
train = pd.read_csv('data/Train.csv')
test = pd.read_csv('data/Test.csv')
features_df = pd.read_csv('data/engineered_team_features.csv')

# 3. MERGE ENGINEERED FEATURES (Using 'country' in Train/Test matching 'team_name' in Feature Store)
train = train.merge(features_df, left_on='country', right_on='team_name', how='left')
test = test.merge(features_df, left_on='country', right_on='team_name', how='left')

# Handle our 4 debutant countries ('Curacao', 'Jordan', 'Cabo Verde', 'Uzbekistan') safely by filling missing stats with 0
fill_cols = ['total_wc_appearances', 'hist_goals_per_game', 'hist_conceded_per_game', 'hist_win_ratio']
for col in fill_cols:
    train[col] = train[col].fillna(0)
    test[col] = test[col].fillna(0)

# 4. ENCODE TOURNAMENT STAGES TO INTEGERS
stage_mapping = {
    'group stage': 0, 'second group stage': 0, 'round of 16': 1,
    'quarter-finals': 2, 'third-place match': 3, 'semi-finals': 3,
    'final': 4
}
inv_stage_mapping = {0: 'group', 1: 'roundof16', 2: 'qf', 3: 'sf', 4: 'runnerup'}

train['target_stage_encoded'] = train['stage_reached'].map(stage_mapping).fillna(0).astype(int)

print(f"[+] Merging complete! Train features shape: {train.shape}")
print(f"[+] Merging complete! Test features shape:  {test.shape}")

[+] Merging complete! Train features shape: (489, 21)
[+] Merging complete! Test features shape:  (48, 10)


In [3]:

# 1. DEFINE A SINGLE UNIFIED FEATURE LIST
features = ['matches_played', 'total_wc_appearances', 'hist_goals_per_game', 'hist_conceded_per_game', 'hist_win_ratio', 'points_per_game', 'modern_goals_per_game' ,  'modern_win_ratio']

for col in fill_cols:
    train[col] = train[col].fillna(0)
    test[col] = test[col].fillna(0)

    
# 2. PRE-PREPARE THE TEST DATAFRAME
# Add the missing 'matches_played' baseline to test so it has the exact same columns as train
test['matches_played'] = 3

# Prepare arrays to hold our local "Out-of-Fold" (OOF) predictions
oof_goals = np.zeros(len(train))
oof_stages = np.zeros(len(train))

# Prepare arrays to hold our final 2026 test predictions
test_goals_preds = np.zeros(len(test))
test_stages_preds = np.zeros((len(test), 5)) 

# 3. INITIALIZE THE STRATIFIED CROSS-VALIDATION
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

print("--- Starting 5-Fold Chained Training Loop with Modern Form ---")

for fold, (train_idx, val_idx) in enumerate(skf.split(train, train['target_stage_encoded'])):
    # Split the data into Training and Validation sets for this specific fold
    X_train, y_train_g, y_train_s = train.iloc[train_idx][features], train.iloc[train_idx]['total_goals'], train.iloc[train_idx]['target_stage_encoded']
    X_val, y_val_g, y_val_s = train.iloc[val_idx][features], train.iloc[val_idx]['total_goals'], train.iloc[val_idx]['target_stage_encoded']
    
    # --- PHASE A: Goals Prediction (Regression) ---
    model_goals = LGBMRegressor(random_state=SEED, n_estimators=100, learning_rate=0.05)
    model_goals.fit(X_train, y_train_g, eval_set=[(X_val, y_val_g)])
    
    # Store local validation predictions
    oof_goals[val_idx] = model_goals.predict(X_val)
    
    # Isolate test features safely now that 'matches_played' exists in test
    X_test_fold = test[features].copy()
    
    # Add this fold's test predictions to our running total
    test_goals_preds += model_goals.predict(X_test_fold) / skf.n_splits
    
    # --- PHASE B: Stage Reached Prediction (Classification with Chaining) ---
    X_train_s = X_train.copy()
    X_train_s['predicted_goals'] = model_goals.predict(X_train)
    
    X_val_s = X_val.copy()
    X_val_s['predicted_goals'] = oof_goals[val_idx]
    
    model_stage = LGBMClassifier(random_state=SEED, n_estimators=100, learning_rate=0.05, class_weight='balanced')
    model_stage.fit(X_train_s, y_train_s, eval_set=[(X_val_s, y_val_s)])
    
    # Store local classification predictions
    oof_stages[val_idx] = model_stage.predict(X_val_s)
    
    # Predict probabilities for the 2026 Test set using the dynamic meta-feature
    X_test_s = X_test_fold.copy()
    X_test_s['predicted_goals'] = test_goals_preds
    test_stages_preds += model_stage.predict_proba(X_test_s) / skf.n_splits

print("\n--- Training Complete! ---")

# 4. CALCULATE LOCAL SCORES
cv_rmse = np.sqrt(mean_squared_error(train['total_goals'], oof_goals))
cv_f1 = f1_score(train['target_stage_encoded'], oof_stages, average='macro')

print(f"Local CV Goals RMSE: {cv_rmse:.4f}")
print(f"Local CV Stage F1:   {cv_f1:.4f}")

--- Starting 5-Fold Chained Training Loop with Modern Form ---
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000197 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 267
[LightGBM] [Info] Number of data points in the train set: 391, number of used features: 8
[LightGBM] [Info] Start training from score 5.565217
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf

In [4]:
# Update your lists to include 'modern_goals_per_game'
train_features = ['matches_played', 'total_wc_appearances', 'hist_goals_per_game', 'hist_conceded_per_game', 'hist_win_ratio', 'modern_goals_per_game']
test_features = ['total_wc_appearances', 'hist_goals_per_game', 'hist_conceded_per_game', 'hist_win_ratio', 'modern_goals_per_game']


# 1. DEFINE FEATURES FOR MODELING
# 'train_features' are what the model learns from historically
train_features = ['matches_played', 'total_wc_appearances', 'hist_goals_per_game', 'hist_conceded_per_game', 'hist_win_ratio']

# 'test_features' are what we actually have available in Test.csv
test_features = ['total_wc_appearances', 'hist_goals_per_game', 'hist_conceded_per_game', 'hist_win_ratio']

# Prepare arrays to hold our local "Out-of-Fold" (OOF) predictions
oof_goals = np.zeros(len(train))
oof_stages = np.zeros(len(train))

# Prepare arrays to hold our final 2026 test predictions
test_goals_preds = np.zeros(len(test))
test_stages_preds = np.zeros((len(test), 5)) 

# 2. INITIALIZE THE STRATIFIED CROSS-VALIDATION
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

print("--- Starting 5-Fold Chained Training Loop (Fixed) ---")

for fold, (train_idx, val_idx) in enumerate(skf.split(train, train['target_stage_encoded'])):
    # Split the data into Training and Validation sets for this specific fold
    X_train, y_train_g, y_train_s = train.iloc[train_idx][train_features], train.iloc[train_idx]['total_goals'], train.iloc[train_idx]['target_stage_encoded']
    X_val, y_val_g, y_val_s = train.iloc[val_idx][train_features], train.iloc[val_idx]['total_goals'], train.iloc[val_idx]['target_stage_encoded']
    
    # --- PHASE A: Goals Prediction (Regression) ---
    model_goals = LGBMRegressor(random_state=SEED, n_estimators=100, learning_rate=0.05)
    model_goals.fit(X_train, y_train_g, eval_set=[(X_val, y_val_g)])
    
    # Store local validation predictions
    oof_goals[val_idx] = model_goals.predict(X_val)
    
    # For the Test set, create a temporary dataframe with a default 'matches_played' of 3
    X_test_fold = test[test_features].copy()
    X_test_fold['matches_played'] = 3 # Baseline group stage games
    
    # Ensure columns match the exact order the model was trained on
    X_test_fold = X_test_fold[train_features]
    
    # Add this fold's test predictions to our running total
    test_goals_preds += model_goals.predict(X_test_fold) / skf.n_splits
    
    # --- PHASE B: Stage Reached Prediction (Classification with Chaining) ---
    # Inject predicted goals as a meta-feature
    X_train_s = X_train.copy()
    X_train_s['predicted_goals'] = model_goals.predict(X_train)
    
    X_val_s = X_val.copy()
    X_val_s['predicted_goals'] = oof_goals[val_idx]
    
    model_stage = LGBMClassifier(random_state=SEED, n_estimators=100, learning_rate=0.05, class_weight='balanced')
    model_stage.fit(X_train_s, y_train_s, eval_set=[(X_val_s, y_val_s)])
    
    # Store local classification predictions
    oof_stages[val_idx] = model_stage.predict(X_val_s)
    
    # Predict probabilities for the 2026 Test set using the dynamic meta-feature
    X_test_s = X_test_fold.copy()
    X_test_s['predicted_goals'] = test_goals_preds
    test_stages_preds += model_stage.predict_proba(X_test_s) / skf.n_splits

print("\n--- Training Complete! ---")

# 3. CALCULATE LOCAL SCORES
cv_rmse = np.sqrt(mean_squared_error(train['total_goals'], oof_goals))
cv_f1 = f1_score(train['target_stage_encoded'], oof_stages, average='macro')

print(f"Local CV Goals RMSE: {cv_rmse:.4f}")
print(f"Local CV Stage F1:   {cv_f1:.4f}")

--- Starting 5-Fold Chained Training Loop (Fixed) ---
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000086 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 166
[LightGBM] [Info] Number of data points in the train set: 391, number of used features: 5
[LightGBM] [Info] Start training from score 5.565217
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGB

In [5]:
# 1. DEFINE THE INVERSE DICTIONARY MAPPING
# Translates numbers back into the exact target categories Zindi expects
inv_stage_mapping = {
    0: 'group', 
    1: 'roundof16', 
    2: 'qf', 
    3: 'sf', 
    4: 'runnerup'
}

# 2. CONSTRUCT THE SUBMISSION FRAMEWORK
submission = pd.DataFrame()
submission['ID'] = test['ID']

# Handle predicted goals: Ensure no negative numbers sneak through via a lower boundary clip
submission['total_goals'] = np.clip(test_goals_preds, 0, None)

# Handle predicted tournament stage: Extract the index of the highest probability
final_numeric_stages = np.argmax(test_stages_preds, axis=1)
submission['Target'] = pd.Series(final_numeric_stages).map(inv_stage_mapping)

# 3. VERIFY STRUCTURAL INTEGRITY BEFORE SAVING
print("--- SUBMISSION COLUMNS CHECK ---")
print(submission.columns.tolist())

print("\n--- FIRST 5 PREDICTIONS PREVIEW ---")
display(submission.head(5))

# 4. EXPORT TO CSV
submission.to_csv('data/baseline_submission.csv', index=False)
print("\n[+] SUCCESS! Your submission file has been saved to 'data/baseline_submission.csv'")

--- SUBMISSION COLUMNS CHECK ---
['ID', 'total_goals', 'Target']

--- FIRST 5 PREDICTIONS PREVIEW ---


,ID,total_goals,Target
0,WC-2026_AUT,2.746776,group
1,WC-2026_BEL,2.533583,group
2,WC-2026_BIH,3.274778,group
3,WC-2026_HRV,3.694557,group
4,WC-2026_CZE,1.414673,qf



[+] SUCCESS! Your submission file has been saved to 'data/baseline_submission.csv'


In [6]:
# 1. TRIPLE-CHECK THE REVERSE DICTIONARY MAPPING
# Translates the model's numbers back into the exact text categories Zindi requires
inv_stage_mapping = {
    0: 'group', 
    1: 'roundof16', 
    2: 'qf', 
    3: 'sf', 
    4: 'runnerup'
}

# 2. CONSTRUCT THE SUBMISSION DATAFRAME
final_submission = pd.DataFrame()
final_submission['ID'] = test['ID']

# Handle predicted goals: Ensure no negative numbers sneak through via an lower clip
final_submission['total_goals'] = np.clip(test_goals_preds, 0, None)

# Handle predicted tournament stage: Extract the index of the highest probability
final_numeric_stages = np.argmax(test_stages_preds, axis=1)
final_submission['Target'] = pd.Series(final_numeric_stages).map(inv_stage_mapping)

# 3. PRINT DIAGNOSTICS FOR AWARD-WINNING INTEGRITY
print("--- SUBMISSION LAYOUT CHECK ---")
print(f"File Dimensions (Should be 48 rows, 3 columns): {final_submission.shape}")
print(f"Column Headers (Should be ID, total_goals, Target): {final_submission.columns.tolist()}")

print("\n--- SAMPLE PREDICTIONS PREVIEW ---")
display(final_submission.head(8))

# Double check for any accidental missing values
print(f"\nMissing values found inside file:\n{final_submission.isnull().sum()}")

# 4. EXPORT THE FINAL COMPOSITE FILE
final_submission.to_csv('data/final_award_winning_submission.csv', index=False)
print("\n[+] EXPEDITION COMPLETE! Your optimized file has been saved as 'data/final_award_winning_submission.csv'")

--- SUBMISSION LAYOUT CHECK ---
File Dimensions (Should be 48 rows, 3 columns): (48, 3)
Column Headers (Should be ID, total_goals, Target): ['ID', 'total_goals', 'Target']

--- SAMPLE PREDICTIONS PREVIEW ---


,ID,total_goals,Target
0,WC-2026_AUT,2.746776,group
1,WC-2026_BEL,2.533583,group
2,WC-2026_BIH,3.274778,group
3,WC-2026_HRV,3.694557,group
4,WC-2026_CZE,1.414673,qf
5,WC-2026_ENG,3.508473,group
6,WC-2026_FRA,2.998981,group
7,WC-2026_DEU,4.917873,group



Missing values found inside file:
ID             0
total_goals    0
Target         0
dtype: int64

[+] EXPEDITION COMPLETE! Your optimized file has been saved as 'data/final_award_winning_submission.csv'


In [8]:
# 1. ENFORCE INVERSE DICTIONARY TARGET MAP
inv_stage_mapping = {
    0: 'group', 
    1: 'roundof16', 
    2: 'qf', 
    3: 'sf', 
    4: 'runnerup'
}

# 2. FRAME COMPOSITE COMPONENT
best_submission = pd.DataFrame()
best_submission['ID'] = test['ID']

# Safe continuous clip: floor goal values to 0
best_submission['total_goals'] = np.clip(test_goals_preds, 0, None)

# Assign argmax indices across the probabilistic matrix elements
final_numeric_stages = np.argmax(test_stages_preds, axis=1)
best_submission['Target'] = pd.Series(final_numeric_stages).map(inv_stage_mapping)

# 3. VERIFY STRUCTURAL OUTPUT FOR FAULTLESS ZINDI SUBMISSIONS
print("--- DOMINANCE PIPELINE INTEGRITY ENGINE ---")
print(f"Total Rows Extracted (Should be 48): {len(best_submission)}")
print(f"Missing Values Discovered inside File: {best_submission.isnull().sum().sum()}")

print("\n--- NEW VALUE FORECAST PREVIEW ---")
display(best_submission.head(8))

# 4. WRITE OUT OVER WORKING DISK
best_submission.to_csv('data/optimized_recency_submission.csv', index=False)
print("\n[+] SUCCESS! New top-tier file written securely to 'data/optimized_recency_submission.csv'")

--- DOMINANCE PIPELINE INTEGRITY ENGINE ---
Total Rows Extracted (Should be 48): 48
Missing Values Discovered inside File: 0

--- NEW VALUE FORECAST PREVIEW ---


,ID,total_goals,Target
0,WC-2026_AUT,2.746776,group
1,WC-2026_BEL,2.533583,group
2,WC-2026_BIH,3.274778,group
3,WC-2026_HRV,3.694557,group
4,WC-2026_CZE,1.414673,qf
5,WC-2026_ENG,3.508473,group
6,WC-2026_FRA,2.998981,group
7,WC-2026_DEU,4.917873,group



[+] SUCCESS! New top-tier file written securely to 'data/optimized_recency_submission.csv'


In [9]:
# 1. ENFORCE INVERSE DICTIONARY TARGET MAP
inv_stage_mapping = {
    0: 'group', 
    1: 'roundof16', 
    2: 'qf', 
    3: 'sf', 
    4: 'runnerup'
}

# 2. FRAME COMPOSITE SUBMISSION
optimized_submission = pd.DataFrame()
optimized_submission['ID'] = test['ID']

# Safe continuous clip: floor goal values to 0
optimized_submission['total_goals'] = np.clip(test_goals_preds, 0, None)

# Assign argmax indices across the probabilistic matrix elements
final_numeric_stages = np.argmax(test_stages_preds, axis=1)
optimized_submission['Target'] = pd.Series(final_numeric_stages).map(inv_stage_mapping)

# 3. VERIFY STRUCTURAL OUTPUT FOR FAULTLESS ZINDI SUBMISSIONS
print("--- SUBMISSION INTEGRITY ENGINE ---")
print(f"Total Rows Extracted (Should be 48): {len(optimized_submission)}")
print(f"Missing Values Discovered inside File: {optimized_submission.isnull().sum().sum()}")

print("\n--- TUNED PREDICTIONS PREVIEW ---")
display(optimized_submission.head(8))

# 4. WRITE OUT OVER WORKING DISK
optimized_submission.to_csv('data/optimized_tuned_submission.csv', index=False)
print("\n[+] SUCCESS! New top-tier file written securely to 'data/optimized_tuned_submission.csv'")

--- SUBMISSION INTEGRITY ENGINE ---
Total Rows Extracted (Should be 48): 48
Missing Values Discovered inside File: 0

--- TUNED PREDICTIONS PREVIEW ---


,ID,total_goals,Target
0,WC-2026_AUT,2.746776,group
1,WC-2026_BEL,2.533583,group
2,WC-2026_BIH,3.274778,group
3,WC-2026_HRV,3.694557,group
4,WC-2026_CZE,1.414673,qf
5,WC-2026_ENG,3.508473,group
6,WC-2026_FRA,2.998981,group
7,WC-2026_DEU,4.917873,group



[+] SUCCESS! New top-tier file written securely to 'data/optimized_tuned_submission.csv'
